In [2]:
import pandas as pd
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt

lyrics = pd.read_csv("lyrics_en_es.csv")
main = pd.read_csv("Spotify_weeklytop200_cleaned.csv")

song_stage = main[["id", "Stage"]]  

df = lyrics.merge(song_stage, on="id", how="inner")
print(df.groupby("Stage")["id"].count())

Stage
Stage 1 (2017-2019)    27303
Stage 2 (2020-2022)    27970
Stage 3 (2023-2025)    26488
Name: id, dtype: int64


In [8]:
def clean_lyrics(text):
    text = str(text).lower()
    text = re.sub(r'\[.*?\]', ' ', text)
    text = re.sub(r'\(.*?\)', ' ', text)
    text = re.sub(r'[^a-záéíóúüñ\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df["lyrics_clean"] = df["lyrics"].apply(clean_lyrics)
df.head(3)[["Title", "Stage", "lyrics_clean"]]

,Title,Stage,lyrics_clean
0,Starboy,Stage 1 (2017-2019),ayy i m tryna put you in the worst mood ah p c...
1,Starboy,Stage 1 (2017-2019),ayy i m tryna put you in the worst mood ah p c...
2,Starboy,Stage 1 (2017-2019),ayy i m tryna put you in the worst mood ah p c...


In [25]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

custom_stops_en = list(ENGLISH_STOP_WORDS) + [
    # Contraction fragments (since apostrophes removed during cleaning)
    "ve", "ain", "ft", "re", "ll", "don", "didn", "doesn",
    "couldn", "wouldn", "shouldn", "wasn", "weren", "isn", "aren",
    # Lyric filler & informal words (1st refinement)
    "yeah", "oh", "just", "ooh", "la", "la la", "oh oh", "uh",
    "ah", "na", "da", "hey", "nah", "aye", "ayy", "woah", "whoa",
    "hm", "mm", "wa", "eh", "feat", "like", "got", "let", "say",
    "wanna", "gonna", "gotta", "cause", "cuz", "em", "til", "tryna",
    "come", "make", "see", "back", "take", "tell", "go", "think",
    "away", "right", "good", "man", "stay", "never", "get", "know","lil"
]


custom_stops = list(ENGLISH_STOP_WORDS) + [
    # Spanish function words
    "que", "la", "el", "en", "de", "los", "las", "un", "una", "por",
    "con", "su", "para", "es", "al", "del", "lo", "le", "se", "me",
    "te", "si", "pero", "más", "como", "ya", "todo", "esta", "yo",
    "mi", "tu", "hay", "fue", "son", "muy", "bien", "cuando", "tiene",
    "ser", "era", "también", "hasta", "desde", "sobre", "sin", "entre",
    "ni", "solo", "tan", "ese", "esa", "aquí", "tú", "usted", "porque",
    "donde", "después", "antes", "algo", "nada", "todos", "otro",
    # Lyric filler words (1st refinement)
    "yeah", "oh", "ooh", "ah", "na", "da", "hey", "nah", "aye", "ayy",
    "woah", "whoa", "uh", "hm", "mm", "gonna", "wanna", "gotta",
    "cause", "cuz", "em", "til", "tryna", "ll", "don", "got",
    "let", "just", "like", "know", "baby", "love", "feat", "wa", "eh",
    "yo", "la",
    # Spanish filler & contracted forms(2nd refinement)
    "pa", "ey", "yeh", "oi", "oi oi", "ve", "ti", "va", "nos", "voy", "soy"
]

In [23]:
stages = ["Stage 1 (2017-2019)", "Stage 2 (2020-2022)", "Stage 3 (2023-2025)"]

def get_stage_docs(lang):
    return [
        " ".join(df[(df["Stage"] == s) & (df["language"] == lang)]["lyrics_clean"])
        for s in stages
    ]

In [24]:
stage_docs_en = get_stage_docs("en")

tfidf_en = TfidfVectorizer(stop_words=custom_stops_en, max_features=5000, ngram_range=(1, 2), min_df=1)
tfidf_matrix_en = tfidf_en.fit_transform(stage_docs_en)

tfidf_df_en = pd.DataFrame(
    tfidf_matrix_en.toarray(),
    columns=tfidf_en.get_feature_names_out(),
    index=stages
)

for stage in stages:
    print(f"\n{stage}:")
    print(tfidf_df_en.loc[stage].nlargest(15).to_string())


Stage 1 (2017-2019):
love     0.435665
baby     0.244511
want     0.210238
time     0.193468
need     0.151811
way      0.149380
feel     0.148257
bitch    0.119559
girl     0.112695
fuck     0.109501
look     0.107180
bad      0.103583
life     0.103314
mind     0.103204
shit     0.099271

Stage 2 (2020-2022):
love     0.430933
baby     0.240552
time     0.200674
want     0.185002
way      0.153486
feel     0.151440
need     0.149260
said     0.122456
day      0.121452
life     0.119624
heart    0.118440
girl     0.116275
night    0.109840
look     0.101592
bad      0.101030

Stage 3 (2023-2025):
love     0.412751
baby     0.279931
time     0.235528
want     0.179471
way      0.163229
feel     0.153104
night    0.140656
said     0.130456
life     0.129142
need     0.128026
day      0.108428
look     0.097079
heart    0.094096
world    0.092947
mind     0.091351


In [26]:
stage_docs_es = get_stage_docs("es")

tfidf_es = TfidfVectorizer(stop_words=custom_stops, max_features=5000, ngram_range=(1, 2), min_df=1)
tfidf_matrix_es = tfidf_es.fit_transform(stage_docs_es)

tfidf_df_es = pd.DataFrame(
    tfidf_matrix_es.toarray(),
    columns=tfidf_es.get_feature_names_out(),
    index=stages
)

for stage in stages:
    print(f"\n{stage}:")
    print(tfidf_df_es.loc[stage].nlargest(15).to_string())


Stage 1 (2017-2019):
quiero     0.262912
mí         0.255965
ella       0.245898
sé         0.199195
quiere     0.175112
qué        0.155490
así        0.152004
tengo      0.144180
está       0.135868
ahora      0.128360
conmigo    0.126483
noche      0.126264
amor       0.114222
eso        0.110859
siempre    0.108787

Stage 2 (2020-2022):
ella       0.269255
qué        0.263182
quiero     0.230496
está       0.189806
mí         0.182800
tengo      0.178853
sé         0.165297
quiere     0.158031
siempre    0.156122
noche      0.138033
estoy      0.134975
eso        0.133934
ahora      0.132741
hace       0.122547
cómo       0.120247

Stage 3 (2023-2025):
qué       0.313551
quiero    0.221927
sé        0.181355
está      0.175534
mí        0.157605
tengo     0.156761
estoy     0.143576
eso       0.136300
otra      0.132400
mami      0.127481
ahora     0.127219
noche     0.125531
ella      0.120059
así       0.120001
bebé      0.117585
